In [52]:
#%pip install numpy
#%pip install pandas
#%pip install matplotlib
#%pip install scipy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [53]:
mapping_bmc = {
    "restart-srv11": "192.168.24.91",
    "restart-srv12": "192.168.24.92",
    "restart-srv13": "192.168.24.93",
    #"restart-srv01": "192.168.24.81"
}

mapping_ip = {
    "restart-srv11": "192.168.103.25",
    "restart-srv12": "192.168.103.22",
    "restart-srv13": "192.168.103.23",
    #"restart-srv01": "192.168.24.81"
}
    

In [54]:
def get_cpu_usage():
    cpu_usage = pd.read_json('cpu_usage.json')
    cpu_usage = cpu_usage["data"]["result"]
    cpu_usage_df = pd.DataFrame()
    for node_data in cpu_usage:
        instance = node_data['metric']['instance']
        if instance == "192.168.24.81":
            continue  # skip this node
        values = node_data['values']  # list of [timestamp, value]
        # {"instance":"192.168.103.22:9100"},"values":[[1768818728.829,"0.012933641975389043"],[1768818788.829,"0.012939814814829198"],[1768818848.829,"0.012733024691071959"],...
        timestamps = [pd.to_datetime(int(ts), unit='s') for ts, val in values]
        cpu_values = [int(float(val) * 100) for ts, val in values]  # Convert to percentage
        node_name = [name for name, ip in mapping_ip.items  () if ip in instance][0]
        temp_df = pd.DataFrame({node_name: cpu_values}, index=timestamps)
        cpu_usage_df = pd.concat([cpu_usage_df, temp_df], axis=1)
    cpu_usage_df = cpu_usage_df.sort_index()
    return cpu_usage_df
        

In [55]:
def get_cpu_cores():
    cpu_cores = pd.read_json('Ncpu.json')
    cpu_cores = cpu_cores["data"]["result"]
    cores_dict = {}
    for node_data in cpu_cores:
        instance = node_data['metric']['instance']
        if instance == "192.168.24.81":
            continue  # skip this node if needed
        value = int(node_data['value'][1])
        node_name = [name for name, ip in mapping_ip.items() if ip in instance][0]
        cores_dict[node_name] = value
    return cores_dict

In [56]:
def get_power_consumption():
    power_data = pd.read_json('power_consumption.json')
    power_data = power_data["data"]["result"]
    power_df = pd.DataFrame()
    for node_data in power_data:
        instance = node_data['metric']['instance']
        if instance == "192.168.24.81":
            continue  # skip this node as it has no power data
        # [{"metric":{"instance":"192.168.24.81"},"values":[[1768818687.155,"227"],[1768818747.155,"210"],...
        # create a df with index as timestamps and column as node name, values as power consumption
        values = node_data['values']
        timestamps = [pd.to_datetime(float(ts), unit='s') for ts, val in values]
        power_values = [int(float(val)) for ts, val in values]  # power consumption in watts
        # find node name from mapping_bmc; fallback to the instance string if not found
        node_name = next((name for name, ip in mapping_bmc.items() if ip in instance), instance)
        # create temporary DataFrame and concatenate
        temp_df = pd.DataFrame({node_name: power_values}, index=timestamps)
        power_df = pd.concat([power_df, temp_df], axis=1)
    power_df = power_df.sort_index()
    return power_df

In [63]:
def merge_cpu_power(cpu_df, power_df):
    #merge cpu_usage and power_consumption on index (timestamps): take only the common timestamps (rounded to the integer minute)
    #round index to the nearest minute
    cpu_df.index = cpu_df.index.round('T')
    power_df.index = power_df.index.round('T')
    merged_df = pd.merge(cpu_df, power_df, left_index=True, right_index=True, suffixes=('_cpu', '_power'))
    return merged_df

In [71]:
def map_usage_to_consumption(merged_df):
    #for each node create a dict mapping the cpu usage percentage to the power consumption. If multiple entries exist for the same usage, keep the median consumption
    usage_consumption_dict = {}
    for node in mapping_ip.keys():
        cpu_col = f"{node}_cpu"
        power_col = f"{node}_power"
        node_data = merged_df[[cpu_col, power_col]].dropna()
        usage_to_consumption = node_data.groupby(cpu_col)[power_col].median().to_dict()
        usage_consumption_dict[node] = usage_to_consumption
    return usage_consumption_dict
 

In [ ]:
def interpolate_usage_consumption(usage_consumption_dict): # interpolate missing cpu usage values and consumption with a log () function. If there is only one data, create the dict with the 100 keys (0-100), keep the original values and add NAN values
    interpolated_dict = {}
    for node, usage_consumption in usage_consumption_dict.items():
        usages = np.array(list(usage_consumption.keys()))
        consumptions = np.array(list(usage_consumption.values()))
        
        if len(usages) < 2:
            # Not enough data to fit a curve, create a dict with NANs
            full_dict = {i: usage_consumption.get(i, np.nan) for i in range(101)}
            interpolated_dict[node] = full_dict
            continue
        
        # Define a logarithmic function for fitting
        def log_func(x, a, b):
            return a * np.log(x + 1) + b  # +1 to avoid log(0)
        
        # Fit the curve
        try:
            params, _ = curve_fit(log_func, usages, consumptions, maxfev=10000)
            a, b = params
            
            # Create full mapping from 0 to 100
            full_dict = {}
            for i in range(101):
                if i in usage_consumption:
                    full_dict[i] = usage_consumption[i]
                else:
                    full_dict[i] = log_func(i, a, b)
            interpolated_dict[node] = full_dict
        except Exception as e:
            print(f"Could not fit curve for node {node}: {e}")
            full_dict = {i: usage_consumption.get(i, np.nan) for i in range(101)}
            interpolated_dict[node] = full_dict
    return interpolated_dict
    

In [92]:
def get_cpu_pdf(cpu_usage):
    #compute the probability density function of cpu usage for each node
    pdf_dict = {}
    for node in mapping_ip.keys():
        usage_series = cpu_usage[node].dropna()
        hist, bin_edges = np.histogram(usage_series, bins=range(0, 102), density=True)
        pdf_dict[node] = (hist, bin_edges)
    return pdf_dict

In [95]:
def get_WAP_metric(interpolated_usage_consumption_dict, workload_pdf):
    """
    Calcola la metrica WAP (Workload-Aware Power) per ogni nodo.
    WAP = ∫[0,100] P(cpu%) × PDF(cpu%) d(cpu%)
    
    Args:
        interpolated_usage_consumption_dict: dict {node: {cpu%: power}}
        workload_pdf: dict {node: (hist, bin_edges)} dalla funzione get_cpu_pdf
    
    Returns:
        dict {node: WAP_value} in Watt
    """
    wap_dict = {}
    
    for node in mapping_ip.keys():
        # Get power consumption mapping (cpu% -> power in Watts)
        power_map = interpolated_usage_consumption_dict.get(node, {})
        
        # Get workload probability density function
        hist, bin_edges = workload_pdf.get(node, (np.array([]), np.array([])))
        
        if len(hist) == 0 or len(power_map) == 0:
            wap_dict[node] = np.nan
            continue
        
        # Calcola l'integrale discreto: sum(P(cpu_i) * PDF(cpu_i) * Δcpu)
        # hist è già normalizzato (density=True), quindi rappresenta la PDF
        # bin_edges ha 102 elementi (0 a 101), hist ha 101 elementi
        
        integral = 0.0
        for i in range(len(hist)):
            cpu_usage = i  # percentuale CPU (centro del bin)
            power = power_map.get(cpu_usage, np.nan)
            
            if np.isnan(power):
                continue
            
            # hist[i] è la densità di probabilità per cpu_usage = i
            # Δcpu = 1 (larghezza del bin)
            integral += power * hist[i] * 1.0
        
        wap_dict[node] = integral
    
    return wap_dict
    
    
    
    


In [101]:
def normalize_area(integral_area, cpu_cores):
    normalized_dict = {}
    
    for node in integral_area.keys():
        wap = integral_area.get(node, np.nan)
        cores = cpu_cores.get(node, 1)  # default 1 per evitare divisione per 0
        
        if np.isnan(wap) or cores == 0:
            normalized_dict[node] = np.nan
        else:
            normalized_dict[node] = wap / cores
    
    return normalized_dict
    

In [103]:
cpu_usage = get_cpu_usage()
cpu_cores = get_cpu_cores()
power_consumption = get_power_consumption()

merged_df = merge_cpu_power(cpu_usage, power_consumption)
usage_consumption_dict = map_usage_to_consumption(merged_df)
interpolated_usage_consumption_dict = interpolate_usage_consumption(usage_consumption_dict)

workload_pdf = get_cpu_pdf(cpu_usage)
integral_area = get_WAP_metric(interpolated_usage_consumption_dict, workload_pdf)
normalized_area = normalize_area(integral_area, cpu_cores)
normalized_area



/var/folders/yz/p40mv9ds62v8qdjv5qv2z6gw0000gn/T/ipykernel_29766/2035236506.py:4: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  cpu_df.index = cpu_df.index.round('T')
/var/folders/yz/p40mv9ds62v8qdjv5qv2z6gw0000gn/T/ipykernel_29766/2035236506.py:5: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  power_df.index = power_df.index.round('T')


{'restart-srv11': np.float64(6.3334996528983),
 'restart-srv12': np.float64(6.46875),
 'restart-srv13': np.float64(6.520833333333333)}